# Week 7 — PyTorch Basics

**Theme:** Getting comfortable with tensors and autograd before building
real neural networks in Week 8.

So far scikit-learn has done all the heavy lifting for us (`.fit()` did
everything). Starting next week, we'll build networks ourselves with
**PyTorch** — so before touching a real network, this week is just about the
two building blocks everything else sits on: **tensors** and **autograd**
(automatic differentiation).

## What is PyTorch?

PyTorch is an open-source machine learning framework widely used for developing and training deep learning models.
PyTorch provides efficient tensor computations, automatic differentiation, and GPU acceleration. Its Python-friendly interface and flexible architecture make it particularly popular in the research community.

## Key Advantages of PyTorch

* **Ease of Use**: Provides an intuitive, Python-friendly interface.
* **GPU Acceleration**: Supports GPU-based computations for faster model training and inference.
* **Tensor Operations**: Provides various tensor operations for efficient numerical computations.
* **Automatic Differentiation**: Automatically computes gradients for neural network training.
* **Flexibility and Extensibility**: Allows users to implement custom layers and neural network architectures.

In [ ]:
import torch

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Tensors

A PyTorch `Tensor` is like a NumPy array that can also track gradients and run
on a GPU. That's really all you need to know to get started.

### Frequently Used Options

* **dtype**: setting data type of the tensor.

* **device**: Specifies the device on which the tensor is stored (e.g., CPU or GPU).

* **requires_grad**: telling the automatic differentiation engine (*Autograd*) to track and record all mathematical operations performed on it. This enables PyTorch to:

  * Dynamically build a computational graph of all operations.
  * Automatically calculate gradients (derivatives, e.g., $\frac{dy}{dx}$) for this tensor when you trigger backpropagation by calling `.backward()` on a scalar loss (like `y.backward()`).
  * Store the calculated gradients directly inside the `.grad` attribute of the tensor (e.g., `x.grad`).

  By default, `requires_grad` is set to `False`. You typically set it to `True` for model parameters (such as weights and biases) that need to be updated during training via gradient descent.

In [ ]:
t1 = torch.tensor([1., 2., 3.], requires_grad=True)
print("1. 직접 생성:", t1)
print("1-1 grad before computation and backward: ", t1.grad)
y = (t1**2).sum()
y.backward()
print("1-2 y (=t1[0]^2 + t1[1]^2 + t1[2]^2):", y.item())
print("1-3 grad after backward: ", t1.grad)

In [ ]:
t1_int = torch.tensor([1,2,3], dtype=torch.int32)
t2 = torch.zeros(2, 3)
t3 = torch.ones(2, 3)
t4 = torch.rand(2, 3)
t5 = torch.randn(2, 3)
t6 = torch.arange(start=0, end=10, step=2)
t7 = torch.empty(4,2)

print("1. t1_int:", t1_int)
print("1. t1_int.dtype:", t1_int.dtype)
print("2. zeros(2, 3):\n", t2)
print("3. ones(2, 3):\n", t3)
print("4. rand(2, 3):\n", t4)
print("5. randn(2, 3):\n", t5)
print("6. arange(0, 10, 2):", t6)
print("7. empty(4,2) : ",t7)

## 2. 작은 예제로 직접 학습시켜보기: autograd로 직선 피팅하기

아직 `nn.Module`이나 optimizer는 쓰지 않습니다 (그건 Week 8부터). 대신 `y = w·x + b`
형태의 직선 하나를 **autograd만으로** 직접 학습시켜봅니다 — 데이터에 가장 잘 맞는
`w`, `b`를 경사하강법(gradient descent)으로 찾는, 모든 신경망 학습의 가장 작은 단위
버전입니다.

매 반복(epoch)마다:
1. **forward**: 현재 `w`, `b`로 예측값을 계산합니다.
2. **loss**: 예측값과 실제값의 평균제곱오차(MSE)를 계산합니다.
3. **backward**: `loss.backward()`로 `w.grad`, `b.grad`를 계산합니다.
4. **update**: `w`, `b`를 gradient의 반대 방향으로 조금씩 이동시키고, 다음 반복을
   위해 gradient를 0으로 초기화합니다.

In [ ]:
torch.manual_seed(0)

n = 50
x = torch.linspace(0, 10, n)
true_w, true_b = 2.0, 1.0
y = true_w * x + true_b + torch.randn(n) * 0.5  # 약간의 noise를 섞은 "실제" 데이터

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.01

loss_history = []
for epoch in range(200):
    pred = w * x + b                    # 1. forward
    loss = ((pred - y) ** 2).mean()     # 2. loss (MSE)
    loss.backward()                     # 3. backward (w.grad, b.grad 계산)

    with torch.no_grad():               # 4. update (이 안에서는 autograd가 추적하지 않음)
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()

    loss_history.append(loss.item())
    if epoch % 50 == 0 or epoch == 199:
        print(f"epoch {epoch:3d}  loss={loss.item():.4f}  w={w.item():.3f}  b={b.item():.3f}")

print(f"\n실제 값: w={true_w}, b={true_b}")
print(f"학습된 값: w={w.item():.3f}, b={b.item():.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(loss_history)
axes[0].set_title("Training Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")

axes[1].scatter(x, y, label="data", alpha=0.6)
with torch.no_grad():
    axes[1].plot(x, w * x + b, color="red", label="learned line")
axes[1].set_title("Learned Line vs. Data")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend()

plt.tight_layout()
plt.show()

학습된 `w`, `b`가 실제 값(2.0, 1.0)에 꽤 가깝게 수렴하는 걸 볼 수 있습니다 — `.fit()`
한 줄 뒤에 숨어있던 과정을 그대로 직접 짜본 것입니다. Week 8부터는 이 네 단계(forward →
loss → backward → update)를 `nn.Module`과 optimizer로 더 깔끔하게 반복하면서, 직선 하나가
아니라 여러 개의 뉴런으로 이루어진 **다층 퍼셉트론(MLP)**을 학습시켜봅니다.

## Try it yourself

1. **learning rate를 바꿔보세요.** `lr = 0.1`과 `lr = 0.0001`로 각각 돌려보고, loss
   곡선이 어떻게 달라지는지 확인하세요 — 너무 크면, 너무 작으면 어떤 문제가 생기나요?
2. **noise를 늘려보세요.** `torch.randn(n) * 0.5`를 `* 3.0`으로 바꾸고, 학습된 `w`, `b`가
   실제 값(2.0, 1.0)에서 얼마나 멀어지는지 확인하세요.
3. **2차 함수를 피팅해보세요.** `y = w*x + b` 대신 `y = w2*x**2 + w1*x + b`로 바꾸고,
   데이터를 `y = 0.5*x**2 - 2*x + 1 + noise`처럼 만들어서 똑같은 방식으로 학습시켜보세요
   — 파라미터가 하나 늘어도 같은 네 단계(forward/loss/backward/update)가 그대로
   적용됩니다.